# 🔗 Clase 1: Introducción a LangChain

## Bienvenido a la Semana 2, Clase 1

En esta clase aprenderás:
- ✅ ¿Por qué necesitamos un orquestador?
- ✅ LangChain: Framework para aplicaciones con LLMs
- ✅ Conceptos: Chains, Runnables, LCEL
- ✅ Paralelización vs ejecución secuencial
- ✅ Integración con FastAPI
- ✅ LangGraph: Introducción a grafos

---

In [1]:
# Instalación
!pip install langchain langchain-anthropic langsmith python-dotenv -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-genai 1.0.1 requires google-generativeai<0.5.0,>=0.4.1, but you have google-generativeai 0.8.3 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough, RunnableParallel

load_dotenv()

# Configurar LangSmith (opcional)
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_PROJECT"] = "taller-ia-semana2"

llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0.7)
print("✅ LangChain configurado con Claude")

✅ LangChain configurado con Claude


## 🤔 Parte 1: ¿Por qué LangChain?

### Problemas sin Orquestador

```python
# ❌ Código repetitivo
response1 = openai.chat.completions.create(...)
response2 = openai.chat.completions.create(...)
response3 = openai.chat.completions.create(...)

# ❌ Difícil de mantener
# ❌ Sin reutilización
# ❌ Difícil de testear
```

### Con LangChain

```python
# ✅ Componentes reutilizables
# ✅ Fácil de componer
# ✅ Debugging integrado
# ✅ Ecosistema completo
```

## 🔗 Parte 2: Chains Básicas

Una **chain** es una secuencia de operaciones.

In [3]:
# Chain simple: Prompt → LLM → Output Parser
prompt = ChatPromptTemplate.from_template("Cuenta un chiste sobre {tema}")
output_parser = StrOutputParser()

# Crear chain usando LCEL (LangChain Expression Language)
chain = prompt | llm | output_parser

# Ejecutar
resultado = chain.invoke({"tema": "programadores"})
print(resultado)

Aquí tienes uno:

Un programador va al supermercado. Su esposa le dice:
- "Compra un litro de leche, y si hay huevos, trae una docena"

El programador regresa a casa con 12 litros de leche.

Su esposa le pregunta: - "¿Por qué compraste 12 litros de leche?"

Y él responde: - "Había huevos" 😄

---

El chiste juega con la lógica de programación, donde el programador interpretó literalmente: "si hay huevos, trae una docena (de leche)" en lugar de entender que se refería a una docena de huevos.


### LCEL: LangChain Expression Language

El operador `|` (pipe) conecta componentes:

```python
chain = component1 | component2 | component3
```

Es como un pipeline de Unix!

In [4]:
# Chain con múltiples pasos
from langchain.prompts import ChatPromptTemplate

# Paso 1: Generar idea
idea_prompt = ChatPromptTemplate.from_template(
    "Dame una idea innovadora para una startup de {industria}"
)

# Paso 2: Analizar la idea
analisis_prompt = ChatPromptTemplate.from_template(
    "Analiza esta idea de startup y dame 3 pros y 3 contras:\n\n{idea}"
)

# Chain completa
chain_idea = idea_prompt | llm | StrOutputParser()
chain_analisis = analisis_prompt | llm | StrOutputParser()

# Ejecutar secuencialmente
idea = chain_idea.invoke({"industria": "educación"})
print("💡 Idea generada:")
print(idea)
print("\n" + "="*80 + "\n")

analisis = chain_analisis.invoke({"idea": idea})
print("📊 Análisis:")
print(analisis)

💡 Idea generada:
## **EcoSkills Academy** 🌱
*"Aprender haciendo un mundo mejor"*

### **Concepto Central**
Una plataforma que combina educación práctica con impacto social real, donde los estudiantes desarrollan habilidades técnicas y blandas mientras resuelven problemas ambientales y sociales de su comunidad.

### **Cómo Funciona**
- **Proyectos Reales**: Los estudiantes trabajan en desafíos propuestos por ONGs, gobiernos locales y empresas B-Corp
- **Mentorías Híbridas**: Combinación de expertos en tecnología, emprendedores sociales y líderes comunitarios
- **Certificaciones con Propósito**: Cada certificación incluye evidencia del impacto social/ambiental generado

### **Elementos Innovadores**
1. **Blockchain de Impacto**: Registro inmutable de los proyectos y su impacto medible
2. **IA Adaptativa**: Personaliza el aprendizaje según las necesidades locales y habilidades del estudiante
3. **Economía Circular Educativa**: Los proyectos exitosos generan ingresos que financian nuevas b

📊 Análisis:
## Análisis de EcoSkills Academy 🌱

### **3 PROS** ✅

**1. Propuesta de Valor Diferenciada y Actual**
- Combina dos tendencias en crecimiento: educación práctica y responsabilidad social
- El enfoque de "aprender resolviendo problemas reales" tiene mayor retención y motivación que la educación tradicional
- Se alinea con la demanda creciente de profesionales con conciencia social y ambiental

**2. Múltiples Fuentes de Ingresos y Sostenibilidad**
- Diversificación de revenue streams: estudiantes, empresas B-Corp, ONGs, gobiernos locales
- El modelo de "economía circular educativa" puede generar un ciclo virtuoso de autofinanciamiento
- Potencial de obtener grants y financiamiento de impacto social

**3. Escalabilidad Global con Adaptación Local**
- El framework es replicable pero flexible para adaptarse a diferentes contextos
- La tecnología (blockchain + IA) permite escalar sin perder la personalización
- Network effects: más participantes = más proyectos = más valor para t

## 🔄 Parte 3: Paralelización vs Secuencial

### Ejecución Secuencial

```
A → B → C → Resultado
```

### Ejecución Paralela

```
    ┌─ A ─┐
    ├─ B ─┤ → Combinar → Resultado
    └─ C ─┘
```

In [5]:
# Ejemplo: Analizar un producto desde múltiples perspectivas
producto = "iPhone 15 Pro"

# Crear prompts para diferentes análisis
prompt_tecnico = ChatPromptTemplate.from_template(
    "Analiza las especificaciones técnicas de {producto}"
)
prompt_precio = ChatPromptTemplate.from_template(
    "Analiza la relación calidad-precio de {producto}"
)
prompt_competencia = ChatPromptTemplate.from_template(
    "Compara {producto} con su competencia"
)

# Ejecutar en paralelo usando RunnableParallel
analisis_paralelo = RunnableParallel(
    tecnico=prompt_tecnico | llm | StrOutputParser(),
    precio=prompt_precio | llm | StrOutputParser(),
    competencia=prompt_competencia | llm | StrOutputParser()
)

import time
inicio = time.time()
resultados = analisis_paralelo.invoke({"producto": producto})
tiempo = time.time() - inicio

print(f"⏱️ Tiempo de ejecución: {tiempo:.2f}s\n")
print("📊 Análisis Técnico:")
print(resultados["tecnico"][:200] + "...\n")
print("💰 Análisis de Precio:")
print(resultados["precio"][:200] + "...\n")
print("🏆 Análisis de Competencia:")
print(resultados["competencia"][:200] + "...")

⏱️ Tiempo de ejecución: 15.38s

📊 Análisis Técnico:
## iPhone 15 Pro - Análisis de Especificaciones Técnicas

### **Procesador y Rendimiento**
- **Chip A17 Pro (3nm)**: Primera generación de chips de Apple en proceso de 3 nanómetros
- **CPU de 6 núcleo...

💰 Análisis de Precio:
## Análisis Calidad-Precio del iPhone 15 Pro

### **Aspectos Positivos**

**Construcción y Materiales**
- Titanio grado 5: más ligero y resistente que el acero inoxidable
- Certificación IP68 para res...

🏆 Análisis de Competencia:
Te comparo el iPhone 15 Pro con sus principales competidores:

## **iPhone 15 Pro vs Samsung Galaxy S24+**

**iPhone 15 Pro ventajas:**
- Ecosistema iOS más integrado
- Mejor optimización software-har...


## 🎯 Parte 4: Runnables Avanzados

### RunnablePassthrough

Pasa datos sin modificar:

In [6]:
# Ejemplo: Mantener el input original
from langchain.schema.runnable import RunnablePassthrough

prompt = ChatPromptTemplate.from_template(
    "Traduce al inglés: {texto}"
)

chain_con_original = {
    "original": RunnablePassthrough(),
    "traduccion": prompt | llm | StrOutputParser()
}

resultado = chain_con_original["traduccion"].invoke({"texto": "Hola mundo"})
print(f"Original: Hola mundo")
print(f"Traducción: {resultado}")

Original: Hola mundo
Traducción: "Hello world"


### RunnableLambda

Ejecuta funciones personalizadas:

In [7]:
from langchain.schema.runnable import RunnableLambda

def contar_palabras(texto: str) -> dict:
    """Cuenta palabras en el texto."""
    return {
        "texto": texto,
        "palabras": len(texto.split()),
        "caracteres": len(texto)
    }

# Definir un prompt para este ejemplo
prompt_chiste = ChatPromptTemplate.from_template("Cuenta un chiste sobre {tema}")

# Chain con función personalizada
chain = (
    prompt_chiste 
    | llm 
    | StrOutputParser() 
    | RunnableLambda(contar_palabras)
)

resultado = chain.invoke({"tema": "inteligencia artificial"})
print(f"Texto: {resultado['texto'][:100]}...")
print(f"Palabras: {resultado['palabras']}")
print(f"Caracteres: {resultado['caracteres']}")

Texto: Aquí tienes uno:

¿Por qué la inteligencia artificial nunca gana en el póker?

Porque siempre tiene ...
Palabras: 42
Caracteres: 244


## 🌐 Parte 5: Integración con FastAPI

Ver el archivo `fastapi_app/main.py` para una implementación completa.

Ejemplo básico:

In [8]:
# Código de ejemplo (no ejecutar en notebook)
ejemplo_fastapi = '''
from fastapi import FastAPI
from langchain_anthropic import ChatAnthropic
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser

app = FastAPI()
llm = ChatAnthropic(model="claude-sonnet-4-20250514")

@app.post("/chat")
async def chat(message: str):
    prompt = ChatPromptTemplate.from_template("{input}")
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({"input": message})
    return {"response": response}
'''

print("📝 Ejemplo de integración con FastAPI:")
print(ejemplo_fastapi)

📝 Ejemplo de integración con FastAPI:

from fastapi import FastAPI
from langchain_anthropic import ChatAnthropic
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser

app = FastAPI()
llm = ChatAnthropic(model="claude-sonnet-4-20250514")

@app.post("/chat")
async def chat(message: str):
    prompt = ChatPromptTemplate.from_template("{input}")
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({"input": message})
    return {"response": response}



## 📊 Parte 6: LangGraph - Introducción

**LangGraph** permite crear flujos más complejos con:
- Decisiones condicionales
- Loops
- Estado compartido

```
LangChain: A → B → C (lineal)
LangGraph: A → B → ¿condición? → C o D (con lógica)
```

Lo veremos en profundidad en Semana 3.

## 💡 Ejercicios Prácticos

In [9]:
# Ejercicio 1: Crea una chain que:
# 1. Genere un título de blog sobre un tema
# 2. Genere 3 subtítulos para ese blog
# 3. Escriba la introducción

# 👉 Tu código aquí
tema = "Machine Learning para principiantes"

# Paso 1: Título
prompt_titulo = ChatPromptTemplate.from_template(
    "Genera un título atractivo para un blog sobre: {tema}"
)

# Paso 2: Subtítulos
prompt_subtitulos = ChatPromptTemplate.from_template(
    "Para el blog titulado '{titulo}', genera 3 subtítulos interesantes"
)

# Paso 3: Introducción
prompt_intro = ChatPromptTemplate.from_template(
    "Escribe una introducción de 2 párrafos para un blog titulado '{titulo}'"
)

# Ejecutar
titulo = (prompt_titulo | llm | StrOutputParser()).invoke({"tema": tema})
print(f"📝 Título: {titulo}\n")

subtitulos = (prompt_subtitulos | llm | StrOutputParser()).invoke({"titulo": titulo})
print(f"📋 Subtítulos:\n{subtitulos}\n")

intro = (prompt_intro | llm | StrOutputParser()).invoke({"titulo": titulo})
print(f"✍️ Introducción:\n{intro}")

📝 Título: Aquí tienes algunas opciones de títulos atractivos para tu blog:

## **Opción Principal:**
**"Machine Learning Sin Misterios: Tu Guía Práctica desde Cero"**

## **Alternativas:**

- **"De Novato a Experto: Machine Learning Explicado Fácil"**
- **"Inteligencia Artificial para Humanos: Aprende ML Paso a Paso"**
- **"Decodificando el Machine Learning: La Guía Definitiva para Principiantes"**
- **"ML Academy: Domina la Inteligencia Artificial sin Complicaciones"**
- **"Algoritmos Amigables: Tu Primer Paso en Machine Learning"**

### ¿Por qué funciona el título principal?

✅ **"Sin Misterios"** - Elimina la intimidación
✅ **"Guía Práctica"** - Promete contenido aplicable
✅ **"Desde Cero"** - Perfecto para principiantes
✅ **Claro y directo** - Fácil de recordar

¿Te gusta alguno de estos o prefieres que desarrolle variaciones de algún estilo en particular?



📋 Subtítulos:
Aquí tienes 3 subtítulos atractivos para complementar tu blog de Machine Learning:

## **Subtítulos Recomendados:**

### **Opción 1 (Enfoque Práctico):**
*"Aprende algoritmos inteligentes con ejemplos reales, proyectos paso a paso y cero matemáticas complicadas"*

### **Opción 2 (Enfoque Transformacional):**
*"Transforma tu carrera profesional dominando la tecnología que está revolucionando el mundo"*

### **Opción 3 (Enfoque Accesible):**
*"Descubre cómo las máquinas aprenden usando un lenguaje simple, ejercicios prácticos y herramientas gratuitas"*

---

## **¿Por qué funcionan estos subtítulos?**

✅ **Complementan** el título principal sin repetir información
✅ **Especifican** qué tipo de contenido encontrarán
✅ **Eliminan objeciones** (sin matemáticas complicadas, herramientas gratuitas)
✅ **Prometen resultados** concretos (proyectos, transformación profesional)
✅ **Mantienen** el tono accesible y amigable

**Mi recomendación:** La Opción 1 combina perfectamente con t

✍️ Introducción:
# Machine Learning Sin Misterios: Tu Guía Práctica desde Cero

¿Te has sentido intimidado por términos como "algoritmos de aprendizaje automático", "redes neuronales" o "inteligencia artificial"? No eres el único. El mundo del Machine Learning puede parecer un territorio exclusivo para genios de la programación y matemáticos con doctorados, pero la realidad es muy diferente. Hoy en día, esta tecnología está tan integrada en nuestra vida cotidiana que la usamos sin darnos cuenta: desde las recomendaciones de Netflix hasta el reconocimiento facial de nuestros smartphones. Es hora de desmitificar esta fascinante disciplina y descubrir que, con la guía adecuada, cualquier persona curiosa puede entender y aplicar sus conceptos fundamentales.

Este blog nace con una misión clara: convertir lo complejo en simple, lo abstracto en práctico, y lo intimidante en accesible. Aquí no encontrarás fórmulas matemáticas incomprensibles ni códigos que requieren años de experiencia para e

In [10]:
# Ejercicio 2: Análisis paralelo de sentimiento
# Analiza el mismo texto desde 3 perspectivas: positivo, negativo, neutral

texto_analizar = "El nuevo producto es innovador pero muy caro."

# 👉 Crea un RunnableParallel que analice desde las 3 perspectivas
analisis_sentimiento = RunnableParallel(
    positivo=ChatPromptTemplate.from_template(
        "Identifica los aspectos POSITIVOS de: {texto}"
    ) | llm | StrOutputParser(),
    negativo=ChatPromptTemplate.from_template(
        "Identifica los aspectos NEGATIVOS de: {texto}"
    ) | llm | StrOutputParser(),
    neutral=ChatPromptTemplate.from_template(
        "Da un análisis OBJETIVO de: {texto}"
    ) | llm | StrOutputParser()
)

resultados = analisis_sentimiento.invoke({"texto": texto_analizar})
for clave, valor in resultados.items():
    print(f"\n{clave.upper()}:")
    print(valor)


POSITIVO:
Los aspectos **POSITIVOS** de la frase "El nuevo producto es innovador pero muy caro" son:

## ✅ **Innovación**
- El producto presenta características novedosas
- Aporta algo nuevo al mercado
- Demuestra creatividad y desarrollo tecnológico
- Puede ofrecer soluciones únicas o mejoradas
- Representa avance e evolución en su categoría

## ✅ **Potencial diferenciación**
- Se distingue de la competencia
- Puede crear una ventaja competitiva
- Atrae a consumidores que buscan novedad

## ✅ **Posible calidad superior**
- El precio alto puede reflejar materiales premium
- Podría indicar mayor durabilidad o rendimiento
- Sugiere inversión en investigación y desarrollo

La **innovación** es claramente el aspecto más destacado y positivo de esta descripción.

NEGATIVO:
Los aspectos **NEGATIVOS** identificados en la frase son:

## **"muy caro"**

**Implicaciones negativas:**
- **Barrera económica** para muchos consumidores potenciales
- **Limitación del mercado objetivo** a segmentos de

## 🎓 Resumen

### Conceptos Clave

1. **LangChain**: Framework para orquestar LLMs
2. **LCEL**: Sintaxis con `|` para conectar componentes
3. **Chains**: Secuencias de operaciones
4. **Runnables**: Componentes ejecutables
5. **Paralelización**: Ejecutar múltiples operaciones simultáneamente
6. **LangSmith**: Debugging y monitoreo

### Próxima Clase

En **Clase 2** aprenderemos:
- 🤖 Agentes que toman decisiones
- 🛠️ Tools (herramientas)
- 🧠 ReAct reasoning
- 🌐 Búsqueda web con Tavily

---

**¡Nos vemos en la próxima clase! 🚀**